In [17]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, desc, sum, avg , udf
spark = SparkSession.builder.appName("SparkCompleteNotes").getOrCreate()
# Create Base DataFrame
data = [
    (1, "Alice", 25),
    (2, "Bob", 30),
    (3, "Charlie",35),
    (4, "David", 28),
    (5, "Eve", 32),
    (6,"Alish",17)
]
columns = ["Id", "Name", "Age"]
df = spark.createDataFrame(data, columns);

In [11]:
def categorize_age(age):
    if age>=18:
        return "Adult"
    return "Minor"
    

In [12]:
age_categ_udf=udf(categorize_age,StringType())

In [14]:
df_method1 = df.withColumn("Category",age_categ_udf(col("age")))
print("Method 1: Standard UDF via Dataframe API")
df_method1.show()

Method 1: Standard UDF via Dataframe API


+---+-------+---+--------+
| Id|   Name|Age|Category|
+---+-------+---+--------+
|  1|  Alice| 25|   Adult|
|  2|    Bob| 30|   Adult|
|  3|Charlie| 35|   Adult|
|  4|  David| 28|   Adult|
|  5|    Eve| 32|   Adult|
+---+-------+---+--------+



In [18]:
def canDrive(age):
    if age>=18:
        return "Can Drive"
    return "Cannot Drive"

In [19]:
canDriveUdf=udf(canDrive,StringType())
df_method2=df.withColumn("Driver Applicable",canDriveUdf(col("age")))
df_method2.show()

+---+-------+---+-----------------+
| Id|   Name|Age|Driver Applicable|
+---+-------+---+-----------------+
|  1|  Alice| 25|        Can Drive|
|  2|    Bob| 30|        Can Drive|
|  3|Charlie| 35|        Can Drive|
|  4|  David| 28|        Can Drive|
|  5|    Eve| 32|        Can Drive|
|  6|  Alish| 17|     Cannot Drive|
+---+-------+---+-----------------+



In [20]:
spark.udf.register("sql_categ_age",categorize_age,StringType())
df.createOrReplaceTempView("people")
sql_df=spark.sql("""
    Select Name , Age , sql_categ_age(Age) AS Category
    FROM people
    """)
sql_df.show()

+-------+---+--------+
|   Name|Age|Category|
+-------+---+--------+
|  Alice| 25|   Adult|
|    Bob| 30|   Adult|
|Charlie| 35|   Adult|
|  David| 28|   Adult|
|    Eve| 32|   Adult|
|  Alish| 17|   Minor|
+-------+---+--------+



In [21]:
spark.stop()